# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasdibe23-sys/Flyrank-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


My lane as an ML task
ML Task Type: Binary Classification (or Score Ranking)

Framing: We frame the Content Refresh problem as a binary classification task (or a score-ranking model) that predicts whether a given page will benefit from a content refresh. Instead of manually inspecting thousands of pages, the model scores each page with a probability between 0 and 1, allowing the editorial team to prioritize their weekly review queue efficiently.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Target Variable Name: target_needs_refresh

Definition & Logic: Since explicit ground-truth flags for "needed a refresh" are not pre-labeled in the historical dataset, we define a supervised proxy target using performance decay metrics:

A page is assigned target_needs_refresh = 1 if it has high historical search demand (impressions_90d > median) but is experiencing recent conversion or traffic decay (clicks_last_30d < clicks_prev_30d or low recent ctr).

Otherwise, it is assigned target_needs_refresh = 0.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Primary Metric: Precision@K (e.g., Precision@50) / PR-AUC (Precision-Recall Area Under Curve)Rationale: The editorial team has limited human bandwidth and can only manually refresh a fixed number $K$ of pages per week (e.g., $K = 50$). Global metrics like overall Accuracy are misleading because most pages do not need immediate intervention. Precision@K measures the proportion of true refresh opportunities within the top $K$ pages recommended by the model, maximizing human ROI.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [1]:
import pandas as pd
import numpy as np

# Load dataset directly from your raw GitHub URL
url = 'https://raw.githubusercontent.com/matiasdibe23-sys/Flyrank-Starter-Notebooks/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)

# 1. Define unit of analysis (1 row = 1 page / content piece)
print(f"Unit of Analysis: 1 row represents 1 unique page (content_id)")
print(f"Total units analyzed: {len(df):,}\n")

# 2. Construct the Target Proxy column
# Logic: High overall impressions (above median) combined with recent 30-day click decay
impressions_threshold = df['impressions_90d'].median()
df['target_needs_refresh'] = (
    (df['impressions_90d'] > impressions_threshold) &
    (df['clicks_last_30d'] < df['clicks_prev_30d'])
).astype(int)

# 3. Display sample dataframe showing the unit of analysis and target proxy
cols_to_show = ['content_id', 'impressions_90d', 'clicks_last_30d', 'clicks_prev_30d', 'ctr', 'target_needs_refresh']
print("Sample DataFrame with Target Proxy:")
display(df[cols_to_show].head(10))

print("\nTarget Proxy Distribution:")
print(df['target_needs_refresh'].value_counts(normalize=True).map('{:.1%}'.format))

Unit of Analysis: 1 row represents 1 unique page (content_id)
Total units analyzed: 30,000

Sample DataFrame with Target Proxy:


,content_id,impressions_90d,clicks_last_30d,clicks_prev_30d,ctr,target_needs_refresh
0,content_304f48230142,3803,2,13,0.76,1
1,content_a1fb4e703a9e,15320,2,1,0.05,0
2,content_9aa793d4d895,12581,1,3,0.09,1
3,content_331d6c4de07b,11751,22,17,0.49,0
4,content_d99b7a2d90ca,19140,10,2,0.13,0
5,content_d4084a4bc775,3970,0,1,0.03,1
6,content_9a34b442b552,20,0,0,0.00,0
7,content_a63219c6e95a,1724,1,0,0.06,0
8,content_5e6c160719bc,32574,9,8,0.09,0
9,content_c27558df2b0c,1240,0,0,0.16,0



Target Proxy Distribution:
target_needs_refresh
0    81.0%
1    19.0%
Name: proportion, dtype: object


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Non-linear Multi-Variable Interactions: A simple static rule (e.g., "If impressions > 5,000 and clicks == 0, refresh") fails to capture complex non-linear relationships across multiple dimensions such as content_age_days, freshness_tier, word_count, trend_direction, and engagement_rate.

Dynamic Adaptation to Scale & Seasonality: Static thresholds break when site-wide traffic fluctuates or when seasonal trends occur. A machine learning model continuously adapts feature weightings to changing conditions.

Continuous Rank Ordering: Rules produce rigid, binary pass/fail buckets. An ML scoring model outputs a continuous probability score, enabling automated, fine-grained rank ordering of pages for editorial workflows.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.